In [1]:
import json
import numpy as np
import faiss
from transformers import AutoTokenizer, AutoModelForSequenceClassification,pipeline
from sentence_transformers import SentenceTransformer,CrossEncoder
from collections import defaultdict
import pandas as pd
import os
import torch

In [2]:
Home_dir="1000_movies"
CSV_FILE="1000_movies"
JSON_FILE="1000_movie_dialogue"

In [3]:
with open(f"{Home_dir}/{JSON_FILE}.json","r",encoding="utf-8") as file:
    data=json.load(file)

In [5]:
semantic_model=SentenceTransformer(
    "BAAI/bge-large-en-v1.5",
)

emotion_model = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None,
    device=0
)

intent_model=pipeline(
    "zero-shot-classification",
    model=AutoModelForSequenceClassification.from_pretrained("facebook/bart-large-mnli"),
    tokenizer=AutoTokenizer.from_pretrained("facebook/bart-large-mnli"),
    device=0
)

reranker=CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8055.20it/s]


In [6]:
INTENT_LABELS = [

    "romantic confession",
    "love",
    "friendship",
    "sadness",
    "grief",
    "revenge",
    "motivation",
    "determination",
    "hope",
    "fear",
    "anger",
    "warning",
    "philosophical reflection",
    "identity crisis",
    "heroism",
    "betrayal",
    "justice",
    "loneliness",
    "self sacrifice",
    "inspiration"
]

In [ ]:
ids=[]
summary_semantic_vectors = []
summary_emotion_vectors = []
summary_intent_vectors = []

text_semantic_vectors = []
text_emotion_vectors = []
text_intent_vectors = []


# SUMMARY SEMANTIC
for i, item in enumerate(data):
    summary = item["summary"]

    summary_semantic = semantic_model.encode(summary,normalize_embeddings=True)
    summary_semantic_vectors.append(summary_semantic)

    if i % 100 == 0:
        print(f"SUMMARY SEMANTIC Processed: {i}")

summary_semantic_vectors=np.array(summary_semantic_vectors,dtype=np.float32)
np.save(f"{Home_dir}/embeddings/summary_semantic.npy",summary_semantic_vectors)
summary_semantic_vectors=np.array([])

# SUMMARY EMOTION
for i, item in enumerate(data):
    summary = item["summary"]
#
    summary_emotion_result = emotion_model(summary)[0]
    summary_emotion_vector = [x["score"]for x in summary_emotion_result]
    summary_emotion_vectors.append(summary_emotion_vector)
    if i % 100 == 0:
        print(f"SUMMARY EMOTION Processed: {i}")

summary_emotion_vectors=np.array(summary_emotion_vectors,dtype=np.float32)
np.save(f"{Home_dir}/embeddings/summary_emotion.npy",summary_emotion_vectors)
summary_emotion_vectors=np.array([])

# SUMMARY INTENT
for i, item in enumerate(data):
    summary = item["summary"]
  
    summary_intent_result = intent_model(summary,INTENT_LABELS)
    summary_intent_vector = (summary_intent_result["scores"])
    summary_intent_vectors.append(summary_intent_vector)

    if i % 100 == 0:
        print(f"SUMMARY INTENT Processed: {i}")

summary_intent_vectors=np.array(summary_intent_vectors,dtype=np.float32)
np.save(f"{Home_dir}/embeddings/summary_intent.npy",summary_intent_vectors)
summary_intent_vectors=np.array([])


# TEXT INTENT
for i, item in enumerate(data):
    text = item["text"]

    text_intent_result = intent_model(text,INTENT_LABELS)
    text_intent_vector = (text_intent_result["scores"])
    text_intent_vectors.append(text_intent_vector)

    if i % 100 == 0:
        print(f"TEXT INTENT Processed: {i}")

text_intent_vectors=np.array(text_intent_vectors,dtype=np.float32)
np.save(f"{Home_dir}/embeddings/text_intent.npy",text_intent_vectors)
text_intent_vectors=np.array([])


# TEXT EMOTION
# for i, item in enumerate(data):
#     text = item["text"]

#     text_emotion_result = emotion_model(text)[0]
#     text_emotion_vector = [x["score"] for x in text_emotion_result]
#     text_emotion_vectors.append(text_emotion_vector)

#     if i % 100 == 0:
#         print(f"TEXT EMOTION Processed: {i}")

# text_emotion_vectors=np.array(text_emotion_vectors,dtype=np.float32)
# np.save(f"{Home_dir}/embeddings/text_emotion.npy",text_emotion_vectors)
# text_emotion_vectors=np.array([])

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
tokenizer=AutoTokenizer.from_pretrained("SamLowe/roberta-base-go_emotions")

MAX_TOKENS = 480

def chunk_text(text):
    if text is None:
        return [""]
    
    text = str(text).strip()
    if len(text) == 0:
        return [""]

    tokens = tokenizer.encode(text,add_special_tokens=False)
    chunks = []
    for i in range(0, len(tokens), MAX_TOKENS):
        chunk_tokens = tokens[i:i + MAX_TOKENS]
        if len(chunk_tokens) == 0:
            continue

        chunk = tokenizer.decode(chunk_tokens,skip_special_tokens=True,clean_up_tokenization_spaces=True)
        chunks.append(chunk)

    if len(chunks) == 0:
        chunks = [""]

    return chunks


text_emotion_vectors = []
for i, item in enumerate(data):

    try:
        text = item.get("text", "")
        chunks = chunk_text(text)
        chunks = [c for c in chunks if isinstance(c, str) and len(c.strip()) > 0]
        if len(chunks) == 0:
            chunks = ["empty"]

        with torch.no_grad():
            chunk_outputs = emotion_model(chunks,truncation=True,padding=True,batch_size=8)

        chunk_score_vectors = []
        for output in chunk_outputs:
            if not isinstance(output, list):
                continue
            scores = []
            for x in output:
                score = float(x["score"])
                if np.isnan(score):
                    score = 0.0

                scores.append(score)

            if len(scores) > 0:
                chunk_score_vectors.append(scores)

        if len(chunk_score_vectors) == 0:
            num_emotions = len(chunk_outputs[0]) \
                if len(chunk_outputs) > 0 else 1

            final_scores = np.zeros(num_emotions,dtype=np.float32)
        else:
            final_scores = np.mean(chunk_score_vectors,axis=0).astype(np.float32)

        text_emotion_vectors.append(final_scores)

        if i % 100 == 0:
            print(f"Processed: {i}/{len(data)}")

    except Exception as e:
        print(f"\nERROR at index {i}")
        print(e)
        text_emotion_vectors.append(np.zeros(6, dtype=np.float32))
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


text_emotion_vectors = np.array(text_emotion_vectors,dtype=np.float32)

np.save(f"{Home_dir}/embeddings/text_emotion.npy",text_emotion_vectors)
print("\nFinal Shape:")
print(text_emotion_vectors.shape)


# TEXT SEMANTIC
for i, item in enumerate(data):
    text = item["text"]

    text_semantic = semantic_model.encode(text,normalize_embeddings=True)
    text_semantic_vectors.append(text_semantic)
    if i % 100 == 0:
        print(f"TEXT SEMANTIC Processed: {i}")

text_semantic_vectors=np.array(text_semantic_vectors,dtype=np.float32)
np.save(f"{Home_dir}/embeddings/text_semantic.npy",text_semantic_vectors)
text_semantic_vectors=np.array([])


print("\nFinished Embedding Creation")

In [ ]:
import numpy as np
import os

# Path to your .npy file
file_path = f"{Home_dir}/embeddings/summary_intent.npy"

# File size on disk
size_bytes = os.path.getsize(file_path)
size_mb = size_bytes / (1024 ** 2)

print(f"File size: {size_bytes} bytes")
print(f"File size: {size_mb:.2f} MB")

# Load array and inspect
arr = np.load(file_path)

print(f"Shape: {arr.shape}")
print(f"Data type: {arr.dtype}")
print(f"Array size (elements): {arr.size}")

# Memory occupied by array
print(f"Memory used by array: {arr.nbytes / (1024 ** 2):.2f} MB")

In [ ]:
res=faiss.StandardGpuResources()


def save_all_indices():
    mappings = {
        f"{Home_dir}/embeddings/summary_semantic.npy": f"{Home_dir}/index/summary_semantic.index",
        f"{Home_dir}/embeddings/summary_emotion.npy": f"{Home_dir}/index/summary_emotion.index",
        f"{Home_dir}/embeddings/summary_intent.npy": f"{Home_dir}/index/summary_intent.index",
        f"{Home_dir}/embeddings/text_semantic.npy": f"{Home_dir}/index/text_semantic.index",
        f"{Home_dir}/embeddings/text_emotion.npy": f"{Home_dir}/index/text_emotion.index",
        f"{Home_dir}/embeddings/text_intent.npy": f"{Home_dir}/index/text_intent.index"
    }
    
    for npy_path, index_path in mappings.items():
        vectors = np.load(npy_path)
        dim = vectors.shape[1]
        cpu_index = faiss.IndexFlatIP(dim)
        cpu_index.add(vectors)
        faiss.write_index(cpu_index, index_path)


In [7]:
def load_gpu_index(index_path):
    cpu_index = faiss.read_index(index_path)
    return cpu_index

In [ ]:
save_all_indices()

In [ ]:
!pip install ollama

In [14]:
from ollama import chat

def summary_ollama(query):
    prompt = f"""
    You are a strict summarizer.

    Rules:
    - Return ONLY the summary
    - No explanations
    - No headings
    - No markdown
    - No extra text
    - Keep it concise

    Text:
    {query}
    """

    response = chat(
        model='gemma4:e2b',
        messages=[
            {
                'role': 'user',
                'content': prompt,
            }
        ]
    )

    summary = response['message']['content']
    return summary



In [30]:
def search_dialogue(query, top_k=10):
    summary=summary_ollama(query)
    print("\n")
    print("=" * 100)
    print("dialogue:", query)
    print("summary:", summary)
    print("=" * 100)

    text_semantic = semantic_model.encode(query, normalize_embeddings=True).astype(np.float32)

    text_emotion_result = emotion_model(query)[0]
    text_emotion = np.array([x["score"] for x in text_emotion_result], dtype=np.float32)

    text_intent_result = intent_model(query, INTENT_LABELS)
    text_intent = np.array(text_intent_result["scores"], dtype=np.float32)

    
    summary_semantic = semantic_model.encode(summary, normalize_embeddings=True).astype(np.float32)

    summary_emotion_result = emotion_model(summary)[0]
    summary_emotion = np.array([x["score"] for x in summary_emotion_result], dtype=np.float32)

    summary_intent_result = intent_model(summary, INTENT_LABELS)
    summary_intent = np.array(summary_intent_result["scores"], dtype=np.float32)

    all_results = defaultdict(float)
    retrieval_scores = {}

    scores, ids_found = load_gpu_index(f"{Home_dir}/index/summary_semantic.index").search(summary_semantic.reshape(1, -1), top_k)
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.20 * float(score)
        all_results[idx] += weighted_score

    # scores, ids_found = load_gpu_index(f"{Home_dir}/index/summary_emotion.index").search(summary_emotion.reshape(1, -1), top_k)
    # for score, idx in zip(scores[0], ids_found[0]):
    #     weighted_score = 0.15 * float(score)
    #     all_results[idx] += weighted_score

    # scores, ids_found = load_gpu_index(f"{Home_dir}/index/summary_intent.index").search(summary_intent.reshape(1, -1), top_k)
    # for score, idx in zip(scores[0], ids_found[0]):
    #     weighted_score = 0.15 * float(score)
    #     all_results[idx] += weighted_score
  
    scores, ids_found = load_gpu_index(f"{Home_dir}/index/text_semantic.index").search(text_semantic.reshape(1, -1), top_k)
    for score, idx in zip(scores[0], ids_found[0]):
        weighted_score = 0.20 * float(score)
        all_results[idx] += weighted_score

    # scores, ids_found = load_gpu_index(f"{Home_dir}/index/text_emotion.index").search(text_emotion.reshape(1, -1), top_k)
    # for score, idx in zip(scores[0], ids_found[0]):
    #     weighted_score = 0.15 * float(score)
    #     all_results[idx] += weighted_score

    # scores, ids_found = load_gpu_index(f"{Home_dir}/index/text_intent.index").search(text_intent.reshape(1, -1), top_k)
    # for score, idx in zip(scores[0], ids_found[0]):
    #     weighted_score = 0.15 * float(score)
    #     all_results[idx] += weighted_score

    candidates = sorted(all_results.items(), key=lambda x: x[1], reverse=True)[:top_k]

    for idx, score in candidates:
        retrieval_scores[idx] = score

    summary_pairs = []
    text_pairs = []
    candidate_ids = []
    for idx, score in candidates:
        item = data[idx]
        summary_pairs.append((query, item["summary"]))
        text_pairs.append((query, item["text"]))
        candidate_ids.append(idx)

    summary_rerank_scores = reranker.predict(summary_pairs)
    text_rerank_scores = reranker.predict(text_pairs)

    final_results = []
    for idx, summary_score, text_score in zip(candidate_ids, summary_rerank_scores, text_rerank_scores):
        retrieval_score = retrieval_scores[idx]
        final_score = (0.30 * retrieval_score + 0.30 * float(summary_score) + 0.40 * float(text_score))
        final_results.append((idx, final_score, retrieval_score, float(summary_score), float(text_score)))

    final_results.sort(key=lambda x: x[1], reverse=True)

    
    csv_file=pd.read_csv(f"{Home_dir}/{CSV_FILE}.csv")
    print("\n")
    print("=" * 100)
    print("FINAL RESULTS")
    print("=" * 100)
    for rank, result in enumerate(final_results):
        idx = result[0]
        final_score = result[1]
        retrieval_score = result[2]
        summary_score = result[3]
        text_score = result[4]
        item = data[idx]
        # print("\n")
        # print("-" * 100)
        # print(f"RANK: {rank + 1}")
        # print(f"FINAL SCORE: {final_score:.4f}")
        # print(f"RETRIEVAL SCORE: {retrieval_score:.4f}")
        # print(f"SUMMARY RERANK: {summary_score:.4f}")
        # print(f"TEXT RERANK: {text_score:.4f}")
        # print(f"ID: {item['_id']}")
        result = csv_file.loc[csv_file['imdb_id'] == item['_id'].split('_')[0],'original_title']
        movie_name = result.iloc[0] if not result.empty else "Not Found"
        print(f"MOVIE_NAME: {movie_name}")
        # print("\nSUMMARY:")
        # print(item["summary"])
        # print("\nTEXT:")
        # print(item["text"])
        # print("-" * 100)

In [12]:
{"dune part one " : "your mother respect me ,you have to  like her. if you move you will die there is an posion stick near it.",
  "dune part one" : "A man tells his father he wanted to become a pilot rather than a leader, but his family believes true leadership is something fate chooses for you.",
  "dune part one" : "i came a dream, where you will die , please dont go",
  "interstellar"  : "you are not belong to this generation , you born too before or after , your wife know it",
  "interstellar"  : "no one want to see there own  die",
  "prisoners"     : "A father abandons ethical boundaries while trying to recover a missing child.",
  "interstellar"  : "don't let me go",
  "Avengers: Endgame" : "i thought by clearing half of this people reamining enjoy it,but you failed me"} 

{'dune part one ': 'your mother respect me ,you have to  like her. if you move you will die there is an posion stick near it.',
 'dune part one': 'i came a dream, where you will die , please dont go',
 'interstellar': "don't let me go",
 'prisoners': 'A father abandons ethical boundaries while trying to recover a missing child.',
 'Avengers: Endgame': 'i thought by clearing half of this people reamining enjoy it,but you failed me'}

In [17]:
dialogues =[
  {
    "movie_name": "Interstellar",
    "dialogue": "Love is the one thing that transcends time and space.",
    "context_dialogue": "Even science cannot explain why people stay emotionally connected no matter how far they are."
  },
  {
    "movie_name": "Inception",
    "dialogue": "An idea is like a virus.",
    "context_dialogue": "Once a thought enters someone mind, it slowly changes everything they do."
  },
  {
    "movie_name": "The Dark Knight",
    "dialogue": "Why so serious?",
    "context_dialogue": "A criminal keeps making fun of people because he thinks society rules are fake."
  },
  {
    "movie_name": "Dune",
    "dialogue": "A great man does not seek to lead.",
    "context_dialogue": "A boy wanted a simple life but his family expects him to become a leader."
  },
  {
    "movie_name": "Whiplash",
    "dialogue": "Not quite my tempo.",
    "context_dialogue": "A strict teacher mentally pressures a student to become perfect."
  },
  {
    "movie_name": "Parasite",
    "dialogue": "They are rich, but still nice.",
    "context_dialogue": "Poor people feel rich people behave differently because money makes life easier."
  },
  {
    "movie_name": "Arrival",
    "dialogue": "If you could see your whole life from start to finish.",
    "context_dialogue": "A woman starts understanding time differently after learning alien language."
  },
  {
    "movie_name": "The Prestige",
    "dialogue": "Are you watching closely?",
    "context_dialogue": "People get distracted by magic tricks and fail to notice the real truth."
  },
  {
    "movie_name": "Memento",
    "dialogue": "Memory can change the shape of a room.",
    "context_dialogue": "A man cannot trust his own memories while searching for revenge."
  },
  {
    "movie_name": "The Social Network",
    "dialogue": "You have part of my attention.",
    "context_dialogue": "A smart programmer becomes successful but slowly loses his close relationships."
  },
  {
    "movie_name": "Blade Runner 2049",
    "dialogue": "Sometimes to love someone.",
    "context_dialogue": "An artificial human starts developing real emotions and feelings."
  },
  {
    "movie_name": "Children of Men",
    "dialogue": "As the sound of the playground faded.",
    "context_dialogue": "Humanity is losing hope because no children are being born anymore."
  },
  {
    "movie_name": "Mad Max Fury Road",
    "dialogue": "Witness me!",
    "context_dialogue": "A dying soldier wants his sacrifice to have meaning and respect."
  },
  {
    "movie_name": "Gladiator",
    "dialogue": "What we do in life echoes in eternity.",
    "context_dialogue": "A warrior believes honorable actions will be remembered forever."
  },
  {
    "movie_name": "Her",
    "dialogue": "The heart is not like a box.",
    "context_dialogue": "A lonely man falls emotionally in love with an AI assistant."
  },
  {
    "movie_name": "Ex Machina",
    "dialogue": "Isn't it strange?",
    "context_dialogue": "A robot intelligently manipulates humans to escape confinement."
  },
  {
    "movie_name": "Joker",
    "dialogue": "You would not get it.",
    "context_dialogue": "A mentally troubled man feels ignored and disconnected from society."
  },
  {
    "movie_name": "Shutter Island",
    "dialogue": "Which would be worse?",
    "context_dialogue": "A man struggles between accepting painful truth or living in illusion."
  },
  {
    "movie_name": "The Revenant",
    "dialogue": "As long as you can still grab a breath.",
    "context_dialogue": "An injured man keeps surviving because he wants revenge."
  },
  {
    "movie_name": "Soul",
    "dialogue": "I am afraid that if I died today.",
    "context_dialogue": "A man realizes life is about enjoying small moments, not only achievements."
  }
]

In [18]:
for dialogue in dialogues:
    dial=dialogue.get("context_dialogue", "")
    search_dialogue(dial)
    print("Actual_movie: ",dialogue.get("movie_name", ""))

    



dialogue: Even science cannot explain why people stay emotionally connected no matter how far they are.
summary: Science cannot explain emotional connection regardless of distance.


FINAL RESULTS
MOVIE_NAME: Match Point
MOVIE_NAME: Terminator Salvation
MOVIE_NAME: Oppenheimer
MOVIE_NAME: John Carter
MOVIE_NAME: Mystic River
MOVIE_NAME: Transformers: Dark of the Moon
MOVIE_NAME: Legend
MOVIE_NAME: Dallas Buyers Club
MOVIE_NAME: The Pianist
MOVIE_NAME: The Internship
Actual_movie:  Interstellar


dialogue: Once a thought enters someone mind, it slowly changes everything they do.
summary: A thought slowly changes everything a person does.


FINAL RESULTS
MOVIE_NAME: Cloud Atlas
MOVIE_NAME: Resident Evil: Apocalypse
MOVIE_NAME: Match Point
MOVIE_NAME: John Carter
MOVIE_NAME: Transformers: Dark of the Moon
MOVIE_NAME: Transformers: Dark of the Moon
MOVIE_NAME: The Pianist
MOVIE_NAME: Legend
MOVIE_NAME: The Internship
MOVIE_NAME: Dallas Buyers Club
Actual_movie:  Inception


dialogue: A c

In [ ]:
search_dialogue(
    "a father cries after watching 23 years of missing videos",
    top_k=10,
)

In [ ]:
load_gpu_index(f"{Home_dir}/index/text_semantic.index")